In [11]:
import sys
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.utils as vutils
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [12]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

In [13]:
current_dir = Path.cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done!")

In [14]:
transformation = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])

In [15]:
train_dataset = datasets.MNIST(
    root=project_root / "data",
    train=True,
    transform=transformation,
    download=True
)

In [16]:
test_dataset = datasets.MNIST(
    root=project_root / "data",
    train=False,
    transform=transformation,
    download=True
)

In [17]:
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [31]:
class Encoder(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        self.l_relu = nn.LeakyReLU()
        self.layer1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=4, padding=1, stride=2)
        self.batchnorm_1 = nn.BatchNorm2d(num_features=32)
        self.layer2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.batchnorm_2 = nn.BatchNorm2d(num_features=64)
        self.layer3 = nn.Conv2d(in_channels=64, out_channels=128, stride=2, padding=0, kernel_size=2)
        self.batchnorm_3 = nn.BatchNorm2d(num_features=128)
        # self.flatten = nn.Flatten()
        self.mu_head = nn.Linear(in_features=128*7*7, out_features=latent_dim)
        self.cov_head = nn.Linear(in_features=128*7*7, out_features=latent_dim)

    def forward(self, X):
        shape = X.shape
        X = self.layer1(X)              #(batch, 32, 14, 14)
        X = self.batchnorm_1(X)
        act1 = self.l_relu(X)
        X = self.layer2(act1)           #(batch, 64, 14, 14)
        X = self.batchnorm_2(X)
        act2 = self.l_relu(X)
        X = self.layer3(act2)           #(batch, 128, 7, 7)
        X = self.batchnorm_3(X)
        X = self.l_relu(X)
        X = X.view(shape[0], -1)        #(batch, 128*7*7)
        mu = self.mu_head(X)
        cov = self.cov_head(X)

        return mu, cov, act1, act2

In [36]:
class Decoder(nn.Module):
    def __init__(self, z_size: int):
        super().__init__()

        self.l_relu = nn.LeakyReLU()
        self.sigmoid = nn.Sigmoid()
        self.layer1 = nn.Linear(in_features=z_size, out_features=128*7*7)
        self.batchnorm_1 = nn.BatchNorm2d(num_features=128)
        self.layer2 = nn.ConvTranspose2d(in_channels=128, out_channels=64, stride=2, padding=1, kernel_size=3, output_padding=1)
        self.batchnorm_2 = nn.BatchNorm2d(num_features=64)
        self.layer3 = nn.ConvTranspose2d(in_channels=64, out_channels=32, stride=1, padding=1, kernel_size=3, output_padding=0)
        self.batchnorm_3 = nn.BatchNorm2d(num_features=32)
        self.layer4 = nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, padding=1, stride=2, output_padding=1)

    def forward(self, X):
        shape = X.shape
        X = self.layer1(X)
        X = X.view(shape[0], 128, 7, 7)
        X = self.batchnorm_1(X)
        X = self.l_relu(X)
        X = self.layer2(X)
        X = self.batchnorm_2(X)
        dact1 = self.l_relu(X)
        X = self.layer3(dact1)
        X = self.batchnorm_3(X)
        dact2 = self.l_relu(X)
        X = self.layer4(dact2)
        X = self.sigmoid(X)

        return X, dact1, dact2

In [37]:
# let's check if everythingis working correctly
latent_dim = 16
encoder = Encoder(latent_dim)
decoder = Decoder(latent_dim)

# test encoder
x = torch.randn(4, 1, 28, 28)
mu, cov, act1, act2 = encoder(x)
print("mu shape:  ", mu.shape)    # (4, 16)
print("act1 shape:", act1.shape)  # (4, 32, ?, ?)
print("act2 shape:", act2.shape)  # (4, 64, ?, ?)

# test decoder
z = torch.randn(4, latent_dim)
out, dact1, dact2 = decoder(z)
print("out shape: ", out.shape)   # (4, 1, 28, 28)
print("dact1 shape:", dact1.shape) # should match act2 channels
print("dact2 shape:", dact2.shape) # should match act1 channels

mu shape:   torch.Size([4, 16])
act1 shape: torch.Size([4, 32, 14, 14])
act2 shape: torch.Size([4, 64, 14, 14])
out shape:  torch.Size([4, 1, 28, 28])
dact1 shape: torch.Size([4, 64, 14, 14])
dact2 shape: torch.Size([4, 32, 14, 14])
